# Resume / Fix Run

Helper script used to continue training from the last good checkpoint after an interrupted run, with a simplified MSE-only training loop and an explicit eval-only mode.

This notebook is a standalone, simplified version of `resume_training.py` and follows the same flow as `4 feb/ADJSCC-CSInet+.ipynb`:
1. dataset
2. AF module
3. ATN module
4. encoder
5. real → complex symbols + power normalisation
6. wireless channel
7. complex → real (C2R)
8. decoder
9. STN
10. training loop


## Imports and seed

In [ ]:
import math
import os
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Config

In [ ]:
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
EPOCHS = 500
TRAIN_SAMPLES = 80000
VAL_SAMPLES = 30000
TRAIN_ITERATIONS = TRAIN_SAMPLES // BATCH_SIZE
VAL_ITERATIONS   = VAL_SAMPLES // BATCH_SIZE

## Dataset

Streams the QuaDRiGa CSI HDF5 files lazily and applies a global per-channel scale computed on the train split.

In [ ]:
def get_train_global_scale(train_path, batch_size=500):
    stats = {
        'dl': {'sum_sq': 0.0, 'count': 0},
        'ul': {'sum_sq': 0.0, 'count': 0}
    }

    with h5py.File(train_path, 'r') as f:
        for key in ['csi_dl', 'csi_ul']:
            dataset = f[key]
            num_samples = dataset.shape[3]
            stat_key = 'dl' if key == 'csi_dl' else 'ul'

            for i in range(0, num_samples, batch_size):
                end = min(i + batch_size, num_samples)
                chunk = dataset[:, :, :, i:end]

                real_part = chunk['real'].astype(np.float32)
                imag_part = chunk['imag'].astype(np.float32)

                stats[stat_key]['sum_sq'] += float(np.sum(real_part ** 2) + np.sum(imag_part ** 2))
                stats[stat_key]['count'] += real_part.size + imag_part.size

    out = {}
    for k in ['dl', 'ul']:
        var = stats[k]['sum_sq'] / max(stats[k]['count'], 1)
        std = np.sqrt(var + 1e-12)
        out[k] = {'std': float(std)}

    print("Train-only scale stats:", out)
    return out

class CSIDataCollaterLazyScaled:
    def __init__(self, train_path, val_path, test_path, stats, batch_size=200, snr_low=-10.0, snr_high=10.0):
        self.batch_size = batch_size
        self.snr_low = snr_low
        self.snr_high = snr_high
        self.stats = stats

        self.paths = {
            'train': train_path,
            'val': val_path,
            'test': test_path
        }

        self.files = {}
        self.datasets = {}
        self.lengths = {}

        for mode, path in self.paths.items():
            f = h5py.File(path, 'r')
            self.files[mode] = f

            if 'csi_dl' not in f.keys() or 'csi_ul' not in f.keys():
                raise KeyError(f"{path} missing 'csi_dl' or 'csi_ul'")

            self.datasets[mode] = {
                'dl': f['csi_dl'],
                'ul': f['csi_ul']
            }
            self.lengths[mode] = f['csi_dl'].shape[3]
            print(f"Initialized {mode} loader: {self.lengths[mode]} samples.")

    def _scale(self, data, std_val):
        return data / (std_val + 1e-8)

    def denormalize(self, scaled_data, mode_key):
        std_v = self.stats[mode_key]['std']
        return scaled_data * (std_v + 1e-8)

    def _process_batch_data(self, batch_arr, mode_key, normalize=True):
        arr_real = batch_arr['real'].astype(np.float32)
        arr_imag = batch_arr['imag'].astype(np.float32)

        std_v = self.stats[mode_key]['std']
        if normalize:
            arr_real = self._scale(arr_real, std_v)
            arr_imag = self._scale(arr_imag, std_v)

        batch_arr_comb = np.stack([arr_real, arr_imag], axis=2)
        batch_arr_comb = np.squeeze(batch_arr_comb)
        batch_arr_comb = np.transpose(batch_arr_comb, (3, 2, 0, 1))
        return batch_arr_comb

    def __call__(self, mode="train"):
        ds_dl = self.datasets[mode]['dl']
        ds_ul = self.datasets[mode]['ul']
        N = self.lengths[mode]

        raw_indices = np.random.choice(N, self.batch_size, replace=False)
        raw_indices.sort()

        batch_dl_raw = ds_dl[:, :, :, raw_indices]
        batch_ul_raw = ds_ul[:, :, :, raw_indices]

        batch_dl = self._process_batch_data(batch_dl_raw, 'dl', normalize=True)
        batch_ul = self._process_batch_data(batch_ul_raw, 'ul', normalize=True)

        batch_snr = np.random.uniform(self.snr_low, self.snr_high, (self.batch_size, 1)).astype(np.float32)

        return (
            torch.from_numpy(batch_dl).float(),
            torch.from_numpy(batch_ul).float(),
            torch.from_numpy(batch_snr).float()
        )

    def close(self):
        for f in self.files.values():
            f.close()

In [ ]:
# Set these paths to the QuaDRiGa CSI .mat files on your machine.
train_file = "train_data.mat"
val_file   = "val_data.mat"
test_file  = "test_data.mat"


In [ ]:
stats = get_train_global_scale(train_file)
dataset = CSIDatasetManager(train_file, val_file, test_file, stats)


## AF Module

Channel-wise SNR-aware feature recalibration: GAP over (H,W), concat with SNR (dB), 2-layer MLP → sigmoid → per-channel scale.

In [ ]:
class AFModule(nn.Module):
    def __init__(self, channels, reduction_ratio=2):
        super(AFModule, self).__init__()
        input_dim = channels + 1
        hidden_dim = max(channels // reduction_ratio, 1)

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, snr):
        batch, channels, height, width = x.size()
        global_feat = F.adaptive_avg_pool2d(x, 1).view(batch, channels)
        context = torch.cat([global_feat, snr], dim=1)

        out = self.fc1(context)
        out = self.relu(out)
        out = self.fc2(out)
        scale_factors = self.sigmoid(out).view(batch, channels, 1, 1)

        return x * scale_factors

## ATN — Analysis Transform Network

Three-layer (or wider, in deeper variants) conv stack with asymmetric strides that compresses the 32×32 angular-delay map to the truncated representation used by the SC-CSI encoder.

In [ ]:
class ATN(nn.Module):
    def __init__(self):
        super(ATN, self).__init__()

        self.conv1 = nn.Conv2d(2, 16, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU()
        self.af1 = AFModule(channels=16)

        self.conv2 = nn.Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.bn2 = nn.BatchNorm2d(16)
        self.prelu2 = nn.PReLU()
        self.af2 = AFModule(channels=16)

        self.conv3 = nn.Conv2d(16, 2, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.bn3 = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.prelu1(x)
        x = self.af1(x, snr)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.prelu2(x)
        x = self.af2(x, snr)

        x = self.conv3(x)
        x = self.bn3(x)
        return x

## Encoder — CSINet+ encoder with AF modules

Two 7×7 conv blocks with AF modules, then a fully-connected layer projects the flattened map to the M-dimensional real-valued codeword.

In [ ]:
class CsiNetPlusEncoderWithAF(nn.Module):
    def __init__(self, compression_ratio=16):
        super(CsiNetPlusEncoderWithAF, self).__init__()

        self.input_channels = 2
        self.height = 32
        self.width = 32
        self.total_elements = self.input_channels * self.height * self.width  # 2048
        self.M = int(self.total_elements / compression_ratio)  # 128

        self.conv1 = nn.Conv2d(2, 2, kernel_size=7, stride=1, padding=3)
        self.bn1 = nn.BatchNorm2d(2)
        self.act1 = nn.LeakyReLU(negative_slope=0.3, inplace=True)
        self.af1 = AFModule(channels=2)

        self.conv2 = nn.Conv2d(2, 2, kernel_size=7, stride=1, padding=3)
        self.bn2 = nn.BatchNorm2d(2)
        self.act2 = nn.LeakyReLU(negative_slope=0.3, inplace=True)
        self.af2 = AFModule(channels=2)

        self.fc = nn.Linear(self.total_elements, self.M)

    def forward(self, x, snr):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act1(out)
        out = self.af1(out, snr)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.act2(out)
        out = self.af2(out, snr)

        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

## Real → complex symbols + power normalisation

Splits the M real outputs into an M/2-length complex vector and rescales it to unit average power per symbol.

In [ ]:
def enc_to_complex_and_normalize(encoder_output):
    k = encoder_output.shape[1] // 2
    real_part = encoder_output[:, :k]
    imag_part = encoder_output[:, k:]
    s = torch.complex(real_part, imag_part)  # [B, k]

    power = torch.mean(s.abs().square(), dim=1, keepdim=True)  # [B, 1]
    s_normalized = s / torch.sqrt(power + 1e-8)
    return s_normalized

## Wireless channel

Differentiable OFDM AWGN channel: picks `k` uplink subcarriers, transmits the complex symbols, adds Gaussian noise scaled to the requested SNR, and applies maximum-ratio combining at the BS.

In [ ]:
class WirelessChannelSimulator(nn.Module):
    def __init__(self, num_bs_antennas=32):
        super().__init__()
        self.Nt = num_bs_antennas

    def forward(self, s, snr_db, h_uplink_raw):
        batch_size, k = s.shape
        device = s.device

        num_subcarriers = h_uplink_raw.shape[2]
        if num_subcarriers < k:
            raise ValueError(f"Uplink channel subcarriers ({num_subcarriers}) fewer than feedback symbols ({k})")

        h_sliced = h_uplink_raw[:, :, :k, :]

        h_real = h_sliced[:, 0, :, :]
        h_imag = h_sliced[:, 1, :, :]
        h_u = torch.complex(h_real, h_imag)

        snr_linear = 10 ** (snr_db / 10.0)
        noise_power = 1.0 / snr_linear
        noise_std = torch.sqrt(noise_power / 2.0).unsqueeze(-1)

        z_real = torch.randn(batch_size, k, self.Nt, device=device) * noise_std
        z_imag = torch.randn(batch_size, k, self.Nt, device=device) * noise_std
        z = torch.complex(z_real, z_imag)

        s_expanded = s.unsqueeze(-1)
        y = h_u * s_expanded + z

        h_norm = torch.norm(h_u, dim=2, keepdim=True)
        w = h_u / (h_norm + 1e-8)
        s_hat = torch.sum(torch.conj(w) * y, dim=2)
        return s_hat

## C2R — Complex → real for the decoder

In [ ]:
class ComplexToReal(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, s_hat):
        real_part = s_hat.real
        imag_part = s_hat.imag
        c_hat = torch.cat((real_part, imag_part), dim=1)
        return c_hat

## Decoder — CSINet+ RefineNet stack

FC → 32×32 feature map, an initial conv block, then a chain of RefineNet residual blocks (each conv block is followed by an AF module).

In [ ]:
class ModifiedRefineNetBlock(nn.Module):
    def __init__(self, channels):
        super(ModifiedRefineNetBlock, self).__init__()

        self.conv1 = nn.Conv2d(channels, 8, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm2d(8)
        self.af1 = AFModule(8)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)
        self.af2 = AFModule(16)

        self.conv3 = nn.Conv2d(16, channels, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(channels)
        self.af3 = AFModule(channels)

    def forward(self, x, snr):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.leaky_relu(out, negative_slope=0.3)
        out = self.af1(out, snr)

        out = self.conv2(out)
        out = self.bn2(out)
        out = F.leaky_relu(out, negative_slope=0.3)
        out = self.af2(out, snr)

        out = self.conv3(out)
        out = self.bn3(out)
        out = torch.tanh(out)
        out = self.af3(out, snr)

        out = identity + out
        return out

In [ ]:
class CsiNetPlusDecoder(nn.Module):
    def __init__(self, input_dim=128, height=32, width=32, channels=2, num_blocks=5):
        super(CsiNetPlusDecoder, self).__init__()

        self.height = height
        self.width = width
        self.channels = channels
        self.flattened_dim = height * width * channels

        self.fc = nn.Linear(input_dim, self.flattened_dim)

        self.initial_conv = nn.Conv2d(channels, channels, kernel_size=7, padding=3)
        self.initial_bn = nn.BatchNorm2d(channels)
        self.initial_af = AFModule(channels)

        self.refinenet_chain = nn.ModuleList(
            [ModifiedRefineNetBlock(channels) for _ in range(num_blocks)]
        )

    def forward(self, x, snr):
        x = self.fc(x)
        x = x.view(-1, self.channels, self.height, self.width)

        x = self.initial_conv(x)
        x = self.initial_bn(x)
        x = F.leaky_relu(x, negative_slope=0.3)
        x = self.initial_af(x, snr)

        for block in self.refinenet_chain:
            x = block(x, snr)

        return x

## STN — Synthesis Transform Network

Mirror image of the ATN: transposed-conv stack that expands the latent back to the 32×32 angular-delay map.

In [ ]:
class STN(nn.Module):
    def __init__(self):
        super(STN, self).__init__()

        self.trans_conv1 = nn.ConvTranspose2d(
            in_channels=2, out_channels=16,
            kernel_size=(3, 3), stride=(2, 1),
            padding=(1, 1), output_padding=(1, 0)
        )
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU(num_parameters=16)
        self.af1 = AFModule(channels=16)

        self.trans_conv2 = nn.ConvTranspose2d(
            in_channels=16, out_channels=16,
            kernel_size=(3, 3), stride=(2, 1),
            padding=(1, 1), output_padding=(1, 0)
        )
        self.bn2 = nn.BatchNorm2d(16)
        self.prelu2 = nn.PReLU(num_parameters=16)
        self.af2 = AFModule(channels=16)

        self.trans_conv3 = nn.ConvTranspose2d(
            in_channels=16, out_channels=2,
            kernel_size=(3, 3), stride=(2, 1),
            padding=(1, 1), output_padding=(1, 0)
        )
        self.bn3 = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        out = self.trans_conv1(x)
        out = self.bn1(out)
        out = self.prelu1(out)
        out = self.af1(out, snr)

        out = self.trans_conv2(out)
        out = self.bn2(out)
        out = self.prelu2(out)
        out = self.af2(out, snr)

        out = self.trans_conv3(out)
        out = self.bn3(out)
        return out

## Build the modules

In [ ]:
atn = ATN().to(device)
encoder = CsiNetPlusEncoderWithAF(compression_ratio=16).to(device)
channel_sim = WirelessChannelSimulator(num_bs_antennas=32).to(device)
c2r = ComplexToReal().to(device)
decoder = CsiNetPlusDecoder(input_dim=encoder.M).to(device)
stn = STN().to(device)

all_params = (list(atn.parameters()) + list(encoder.parameters())
              + list(decoder.parameters()) + list(stn.parameters()))
print('Total trainable parameters:', sum(p.numel() for p in all_params if p.requires_grad))


## Training loop

In [ ]:
def save_checkpoint(epoch, optimizer, scheduler, best_val_loss, path):
    checkpoint = {
        'epoch': epoch,
        'atn_state_dict': atn.state_dict(),
        'encoder_state_dict': encoder.state_dict(),
        'decoder_state_dict': decoder.state_dict(),
        'stn_state_dict': stn.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_val_loss,
        'stats': stats
    }
    torch.save(checkpoint, path)
    print(f"  >> Checkpoint saved: {path}")

### Optimiser, scheduler and loss weights

These cells reproduce the variant's exact training recipe — open the source `.py` for the line-by-line argparse / CLI logic.

In [ ]:
# optimizer = optim.Adam(all_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
#                                                   factor=0.5, patience=cfg.patience,
#                                                   min_lr=cfg.min_lr)
# mse_criterion = nn.MSELoss()
# (See the .py for any variant-specific overrides — e.g. cosine LR
#  schedules, AdamW, or per-parameter-group weight decay.)


### Run training

```python
for epoch in range(cfg.epochs):
    train_metrics = run_epoch('train', epoch_index=epoch)
    val_metrics   = run_epoch('val',   epoch_index=epoch)
    # scheduler.step(val_metrics['linear_nmse'])
```

After training, sweep test NMSE over a fixed SNR grid:

```python
snr_points = [-10, -5, 0, 5, 10]
nmse_db = evaluate_snr_sweep(snr_points, split='test')
```